# XGBoost

In [6]:
from xgboost import XGBRegressor
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.impute import SimpleImputer
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV

In [7]:
df_model = pd.read_csv("/Users/murphy.chen/Desktop/data1030final_project/data/Cleaned/cleaned_car_data.csv")

# define features and target variable
y = df_model["price"]
X = df_model.drop(columns=["price"])

# First split: 20% test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

# Second split: from remaining 80%, create 60/20
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

# feature list
numeric_features = ["mileage", "engine_size", "seat_num", "door_num", "car_age"]
categorical_features = [
    "gearbox",
    "fuel_type_clean",
    "bodytype_clean",
    "color_clean",
    "brand_model"
]

# pipelines for numeric and categorical features
cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy = "constant", fill_value="Unknown")),
    ("ohe", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", cat_pipeline, categorical_features),
    ],
    remainder="drop"
)

Train: (320532, 10)
Validation: (106844, 10)
Test: (106844, 10)


In [23]:
# Baseline XGBoost model
xgb_model = XGBRegressor(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_model)
])

# fit on train, validate on validation set
xgb_pipeline.fit(X_train, y_train)

y_val_pred = xgb_pipeline.predict(X_val)

mae = mean_absolute_error(y_val, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
r2 = r2_score(y_val, y_val_pred)

print("=== Baseline XGBoost (Validation) ===")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")

=== Baseline XGBoost (Validation) ===
MAE:  2,359.66
RMSE: 9,528.38
R²:   0.8763


In [24]:
# hypterparameter tuning with GridSearchCV
param_grid = {
    "model__learning_rate": [0.03],
    "model__n_estimators": [8000, 10000],
    "model__max_depth": [6, 8], 
    "model__min_child_weight": [3],
    "model__subsample": [0.7, 0.8],
    "model__colsample_bytree": [0.8, 1.0],
    "model__reg_alpha": [0, 0.1],
    "model__reg_lambda": [1, 5],
    "model__missing": [np.nan],
}

# define the base model
xgb_base = XGBRegressor(
    objective="reg:squarederror",
    tree_method="hist",
    n_jobs=-1,
    random_state=42
)

# define the pipeline
tune_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", xgb_base)
])

# define the gridsearchCV
xgb_search = GridSearchCV(
    estimator=tune_pipeline,
    param_grid=param_grid,
    scoring="neg_mean_absolute_error",
    cv=3,
    verbose=3,
    n_jobs=-1
)

# fit the grid searchCV
xgb_search.fit(X_train, y_train)

print("Best params:")
print(xgb_search.best_params_)

print("\nBest CV MAE:", -xgb_search.best_score_)

Fitting 3 folds for each of 64 candidates, totalling 192 fits
[CV 1/3] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=6, model__min_child_weight=3, model__missing=nan, model__n_estimators=8000, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.7;, score=-1771.332 total time= 3.1min
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=6, model__min_child_weight=3, model__missing=nan, model__n_estimators=8000, model__reg_alpha=0, model__reg_lambda=5, model__subsample=0.7;, score=-1999.926 total time= 3.1min
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=6, model__min_child_weight=3, model__missing=nan, model__n_estimators=8000, model__reg_alpha=0, model__reg_lambda=1, model__subsample=0.7;, score=-1910.742 total time= 3.3min
[CV 2/3] END model__colsample_bytree=0.8, model__learning_rate=0.03, model__max_depth=6, model__min_child_weight=3, model__missing=nan, model__n_estimators

In [26]:
# get the best pipeline
best_model = xgb_search.best_estimator_

# fit on train and val
best_model.fit(X_train_val, y_train_val)

# evaluate on test set
y_test_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2 = r2_score(y_test, y_test_pred)

print("\n=== Final Tuned XGBoost (Test Set) ===")
print(f"MAE:  {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²:   {r2:.4f}")


=== Final Tuned XGBoost (Test Set) ===
MAE:  1,239.82
RMSE: 2,714.37
R²:   0.9863


In [ ]:
# best index in the grid search for XGBoost
best_idx = xgb_search.best_index_

cv_mae_mean = -xgb_search.cv_results_["mean_test_score"][best_idx]
cv_mae_std = xgb_search.cv_results_["std_test_score"][best_idx]

print("=== XGBoost Cross-Validation MAE ===")
print(f"Mean MAE: {cv_mae_mean:,.2f}")
print(f"Std MAE:  {cv_mae_std:,.2f}")

=== XGBoost Cross-Validation MAE ===
Mean MAE: 1,513.42
Std MAE:  51.24


# Elastic Net

In [8]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import ElasticNet
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import warnings
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)



In [9]:
# define features and target variable
y = df_model["price"]
X = df_model.drop(columns=["price"])

# feature lists
numeric_features = ["mileage", "engine_size", "seat_num", "door_num", "car_age"]
categorical_features = [
    "gearbox",
    "fuel_type_clean",
    "bodytype_clean",
    "color_clean",
    "brand_model"
]

# build numeric and categorical transformers
numeric_transformer = Pipeline(
    steps=[
        (
            "imputer",
            IterativeImputer(
                estimator=RandomForestRegressor(
                    n_estimators=1,
                    random_state=42
                ),
                random_state=42,
                max_iter=20,
                tol =1e-2
            )
        ),
        ("scaler", StandardScaler())
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

# column transformer
preprocessor_enet = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# elastic net pipeline
enet_model = ElasticNet(max_iter=20000, random_state=42)

enet_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor_enet),
        ("model", enet_model)
    ]
)

In [10]:
# baseline, always predict the mean of y_train_val
baseline_pred = np.full_like(y_test, y_train_val.mean())

baseline_mae  = mean_absolute_error(y_test, baseline_pred)
baseline_rmse = np.sqrt(mean_squared_error(y_test, baseline_pred))
baseline_r2   = r2_score(y_test, baseline_pred)

print("\n=== Baseline (Mean Predictor) ===")
print(f"MAE:  {baseline_mae:,.2f}")
print(f"RMSE: {baseline_rmse:,.2f}")
print(f"R²:   {baseline_r2:.4f}")


=== Baseline (Mean Predictor) ===
MAE:  10,972.35
RMSE: 23,208.60
R²:   -0.0000


In [14]:
# hyperparameter tuning for ElasticNet
from sklearn.model_selection import RandomizedSearchCV

param_grid_enet = {
    "model__alpha": [0.01, 0.05, 0.1, 0.3, 1, 3, 10],
    "model__l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9]
}


# grid search CV for ElasticNet
enet_search = GridSearchCV(
    estimator=enet_pipeline,
    param_grid=param_grid_enet,
    scoring="neg_mean_absolute_error",
    cv=3,
    n_jobs=-1,
    verbose=2
)

# fit the grid search CV on train data
enet_search.fit(X_train, y_train)

print("Best Elastic Net params:")
print(enet_search.best_params_)

print("\nBest CV MAE:", -enet_search.best_score_)

Fitting 3 folds for each of 35 candidates, totalling 105 fits
[CV] END .............model__alpha=0.01, model__l1_ratio=0.1; total time=  44.5s
[CV] END .............model__alpha=0.01, model__l1_ratio=0.1; total time=  55.0s
[CV] END .............model__alpha=0.01, model__l1_ratio=0.3; total time=  58.3s
[CV] END .............model__alpha=0.01, model__l1_ratio=0.3; total time= 1.1min
[CV] END .............model__alpha=0.01, model__l1_ratio=0.1; total time= 1.1min
[CV] END .............model__alpha=0.01, model__l1_ratio=0.3; total time= 1.3min
[CV] END .............model__alpha=0.05, model__l1_ratio=0.1; total time=  14.5s
[CV] END .............model__alpha=0.05, model__l1_ratio=0.1; total time=  18.6s
[CV] END .............model__alpha=0.05, model__l1_ratio=0.3; total time=  15.2s
[CV] END .............model__alpha=0.05, model__l1_ratio=0.1; total time=  18.1s
[CV] END .............model__alpha=0.01, model__l1_ratio=0.5; total time= 1.6min
[CV] END .............model__alpha=0.01, model_

/opt/anaconda3/envs/data1030/lib/python3.12/site-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


[CV] END .............model__alpha=0.05, model__l1_ratio=0.5; total time=  20.2s
[CV] END .............model__alpha=0.05, model__l1_ratio=0.7; total time=  26.9s
[CV] END .............model__alpha=0.01, model__l1_ratio=0.7; total time= 2.1min
[CV] END .............model__alpha=0.05, model__l1_ratio=0.7; total time=  28.9s
[CV] END ..............model__alpha=0.1, model__l1_ratio=0.1; total time=  10.7s
[CV] END .............model__alpha=0.05, model__l1_ratio=0.7; total time=  28.4s
[CV] END .............model__alpha=0.01, model__l1_ratio=0.7; total time= 2.4min
[CV] END ..............model__alpha=0.1, model__l1_ratio=0.1; total time=  13.6s
[CV] END .............model__alpha=0.01, model__l1_ratio=0.7; total time= 2.4min
[CV] END ..............model__alpha=0.1, model__l1_ratio=0.3; total time=  11.4s
[CV] END ..............model__alpha=0.1, model__l1_ratio=0.1; total time=  12.0s
[CV] END ..............model__alpha=0.1, model__l1_ratio=0.3; total time=  14.7s
[CV] END ..............model

In [15]:
# get the best pipeline
best_enet_pipeline = enet_search.best_estimator_


# refit on train and val
best_enet_pipeline.fit(X_train_val, y_train_val)

y_test_pred_enet = best_enet_pipeline.predict(X_test)

enet_mae = mean_absolute_error(y_test, y_test_pred_enet)
enet_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_enet))
enet_r2 = r2_score(y_test, y_test_pred_enet)

print("\n=== Final Tuned ElasticNet (Test Set) ===")
print(f"MAE:  {enet_mae:,.2f}")
print(f"RMSE: {enet_rmse:,.2f}")
print(f"R²:   {enet_r2:.4f}")


=== Final Tuned ElasticNet (Test Set) ===
MAE:  5,896.75
RMSE: 15,912.27
R²:   0.5299
